# Pure-GNN v3.1 Technical Preflight & Research Runner

This notebook orchestrates the isolated technical preflight, contract verification,
runtime benchmarking, and scaffolded research runner for **Pure-GNN v3.1**.

### Core Governance Mandates
- **Scientific Status**: Falsification platform for raw pixel graph learning.
- **Strict Data Isolation**: Evaluates and splits ONLY from official `train.csv`. Official `test.csv` and `val.csv` are strictly OFF-LIMITS.
- **Scientific Training**: Disabled by default (`RUN_RESEARCH_SCREEN = False`). This task authorizes only implementation and technical preflight.


## 1. User Configuration

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Irthn1311/FER2013_Graph.git"
REPO_BRANCH = "main"
EXPECTED_COMMIT = None

# FER2013 Official Dataset (TRAIN ONLY)
FER_SPLIT_ROOT = Path("/kaggle/input/datasets/doduyquynii/fer13-split/fer13-split")
FER_TRAIN_CSV = FER_SPLIT_ROOT / "train.csv"

# Output directories
OUTPUT_ROOT = Path("/kaggle/working/outputs/pure_gnn_v31/preflight")
PACKAGE_RELATIVE = Path("research/pure_gnn_v31")

# Execution Switches
RUN_TESTS = True
RUN_TECHNICAL_PREFLIGHT = True
RUN_BENCHMARKS = True
RUN_RESEARCH_SCREEN = False  # MUST DEFAULT FALSE (Scientific training NOT authorized)

# Experimental condition: G0, G0.5, G1, G2, G3
CONDITION = "G1"
SEED = 42
BATCH_SIZE = 32
DEVICE_POLICY = "gpu"
ARCHIVE_OUTPUT = True

print("Configured Pure-GNN v3.1 Runner:")
print(f"  Condition: {CONDITION}")
print(f"  Run Tests: {RUN_TESTS}")
print(f"  Run Preflight: {RUN_TECHNICAL_PREFLIGHT}")
print(f"  Run Research Screen (Scientific Training): {RUN_RESEARCH_SCREEN}")


## 2. Clone and Source Validation

In [ ]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

WORKING = Path("/kaggle/working")
PROJECT_PATH = WORKING / "FER2013_Graph"

def run_checked(command, cwd=None, env=None, capture=False):
    actual = [str(item) for item in command]
    display = [re.sub(r"(https://x-access-token:)[^@]+@", r"\1***@", item) for item in actual]
    print("$", " ".join(display))
    result = subprocess.run(
        actual, cwd=cwd, env=env, text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
    )
    if result.returncode:
        if capture and result.stdout:
            print("\n".join(result.stdout.splitlines()[-100:]))
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result.stdout if capture else ""

# If running in Kaggle and repository not cloned yet
if not PROJECT_PATH.exists() and not Path("research/pure_gnn_v31").exists():
    run_checked(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, PROJECT_PATH])
    os.chdir(PROJECT_PATH)
elif PROJECT_PATH.exists():
    os.chdir(PROJECT_PATH)

PACKAGE_PATH = Path("research/pure_gnn_v31").resolve()
if not PACKAGE_PATH.is_dir():
    raise FileNotFoundError(f"Pure-GNN v3.1 package directory not found: {PACKAGE_PATH}")

actual_commit = run_checked(["git", "rev-parse", "HEAD"], capture=True).strip()
print("Repository Head Commit:", actual_commit)
print("Pure-GNN v3.1 Package Path:", PACKAGE_PATH)


## 3. Environment Inspection

In [ ]:
import importlib.metadata
import platform
import tensorflow as tf

print("Python Version:", sys.version)
print("Platform:", platform.platform())
print("TensorFlow Version:", tf.__version__)
print("Physical Devices:", tf.config.list_physical_devices())
print("GPU Devices:", tf.config.list_physical_devices("GPU"))
print("Disk Free GiB:", round(shutil.disk_usage(os.getcwd()).free / (2**30), 2))


## 4. Dependency Installation

In [ ]:
# Install pure_gnn_v31 in editable mode without pulling legacy dependencies
run_checked([sys.executable, "-m", "pip", "install", "-q", "-e", str(PACKAGE_PATH), "--no-deps"])
print("pure_gnn_v31 installed successfully in isolated environment.")


## 5. Import Isolation

In [ ]:
import pure_gnn_v31
pkg_file = Path(pure_gnn_v31.__file__).resolve()
print("pure_gnn_v31 location:", pkg_file)
if PACKAGE_PATH not in pkg_file.parents:
    raise RuntimeError(f"pure_gnn_v31 resolved outside expected root: {pkg_file}")

# Audit loaded modules for forbidden legacy architecture packages
for mod in sys.modules:
    if "lap_gnn" in mod or "ws_hpg" in mod:
        raise RuntimeError(f"Forbidden legacy architecture module detected in runtime: {mod}")
print("Import isolation verified: 0 legacy architecture dependencies loaded.")


## 6. Bounded Tests

In [ ]:
if RUN_TESTS:
    print("Running full Pure-GNN v3.1 test suite...")
    test_output = run_checked(
        [sys.executable, "-m", "pytest", str(PACKAGE_PATH / "tests"), "-q"],
        cwd=str(PACKAGE_PATH),
        capture=True,
    )
    print("\n".join(test_output.splitlines()[-25:]))
else:
    print("Skipping bounded tests (RUN_TESTS=False).")


## 7. FER Train Input & Research Split Validation

In [ ]:
from pure_gnn_v31.data import create_research_split_manifest, assert_not_test_access

if FER_TRAIN_CSV.is_file():
    assert_not_test_access(FER_TRAIN_CSV)
    split_manifest_path = OUTPUT_ROOT / "research_split_manifest.json"
    manifest = create_research_split_manifest(
        train_csv_path=FER_TRAIN_CSV,
        seed=SEED,
        dev_ratio=0.15,
        output_manifest_path=split_manifest_path,
    )
    print("Research Split Created:")
    print(f"  Source: {manifest['source_train_csv']}")
    print(f"  Train Samples: {manifest['research_train_count']}")
    print(f"  Dev Samples: {manifest['research_dev_count']}")
    print(f"  Manifest SHA256: {manifest['manifest_sha256']}")
else:
    print(f"FER Train CSV not found at {FER_TRAIN_CSV}. Skipping dataset split validation.")


## 8. Pure-GNN v3.1 Technical Preflight & Benchmarks

In [ ]:
import json
from pure_gnn_v31.cli.preflight import run_technical_preflight
from pure_gnn_v31.tools.benchmark_runtime import benchmark_batch_sizes

preflight_json = OUTPUT_ROOT / "preflight_report.json"
benchmark_json = OUTPUT_ROOT / "benchmark_report.json"

if RUN_TECHNICAL_PREFLIGHT:
    preflight_report = run_technical_preflight(output_path=str(preflight_json))
    print("Preflight Status:", preflight_report["status"])
    if preflight_report["status"] != "PASS":
        raise RuntimeError(f"Technical preflight failed: {preflight_report}")

if RUN_BENCHMARKS:
    benchmark_report = benchmark_batch_sizes(
        batch_sizes=[16, 32, 64],
        condition=CONDITION,
        output_path=str(benchmark_json),
    )
    print("Benchmark Results:", json.dumps(benchmark_report["benchmarks"], indent=2))


## 9. Research Screen Runner (Scaffolded Only)

In [ ]:
if RUN_RESEARCH_SCREEN:
    raise PermissionError(
        "SCIENTIFIC TRAINING IS NOT AUTHORIZED IN THIS TASK. "
        "Pure-GNN v3.1 is in implementation and technical preflight phase."
    )
else:
    print("Research Screen Runner is inactive (RUN_RESEARCH_SCREEN=False).")
    print("Ready for later authorized scientific comparison: G0 vs G0.5 vs G1.")


## 10. Post-run Validation

In [ ]:
import json

print("=== Post-Run Validation ===")
# 1. Verify preflight report exists and is PASS
if preflight_json.is_file():
    pf = json.loads(preflight_json.read_text(encoding="utf-8"))
    assert pf["status"] == "PASS", "Preflight status is not PASS"
    print("Check 1: Preflight Status PASS [OK]")

# 2. Verify parameter contract
if preflight_json.is_file():
    share = pf["parameter_info"]["coarse_gate_parameter_share"]
    assert share <= 0.005, f"Gate parameter share {share} > 0.005"
    print(f"Check 2: Parameter Contract (gate share = {share * 100:.3f}% <= 0.5%) [OK]")

# 3. Verify no test data was created or accessed
for f in OUTPUT_ROOT.glob("**/*"):
    if "test" in f.name.lower():
        raise RuntimeError(f"Unexpected output file referencing test: {f}")
print("Check 3: Strict Test Isolation Maintained [OK]")


## 11. Evidence Archival

In [ ]:
import tarfile

if ARCHIVE_OUTPUT and OUTPUT_ROOT.exists():
    archive_path = WORKING / "pure_gnn_v31_preflight_evidence.tar.gz"
    with tarfile.open(archive_path, "w:gz") as tar:
        tar.add(OUTPUT_ROOT, arcname="preflight_evidence")
    print(f"Evidence archive created: {archive_path} ({round(archive_path.stat().st_size / 1024, 2)} KiB)")
else:
    print("Archival skipped.")
